# Pipeline comparaison achat/location avec projection quartier

Objectif: produire une sortie finale quartier estimee pour la cartographie.

In [1]:
import pandas as pd
from pathlib import Path
import re
import unicodedata

ROOT_DIR = Path.cwd().resolve().parents[1]
if not (ROOT_DIR / 'data').exists():
    ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / 'data').exists():
    raise FileNotFoundError("Impossible de localiser le dossier data du projet")


def normalize_text(text: str) -> str:
    text = str(text).strip().lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return text.strip('_')


def pick_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    normalized_map = {normalize_text(col): col for col in df.columns}
    for candidate in candidates:
        key = normalize_text(candidate)
        if key in normalized_map:
            return normalized_map[key]
    for candidate in candidates:
        key = normalize_text(candidate)
        for normalized_col, original_col in normalized_map.items():
            if key in normalized_col:
                return original_col
    return None


def to_numeric(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace('\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    return pd.to_numeric(cleaned, errors='coerce')


In [ ]:
# 1) Base annuelle intermediaire pour construire la sortie quartier

# Referentiel quartiers (pour la projection en etape 2)
quartiers_path = ROOT_DIR / 'data' / 'bronze' / 'commun' / 'quartier_paris.csv'
quartiers = pd.read_csv(quartiers_path, sep=';', encoding='utf-8-sig')
quartiers = quartiers.rename(
    columns={
        'C_QUINSEE': 'code_insee_quartier',
        'L_QU': 'nom_quartier',
        'C_AR': 'arrondissement_num',
        'SURFACE': 'surface_quartier_m2',
        'Geometry X Y': 'geometry_xy'
    }
)
quartiers['arrondissement'] = (
    pd.to_numeric(quartiers['arrondissement_num'], errors='coerce')
    .astype('Int64')
    .astype(str)
    .str.zfill(2)
)
quartiers['surface_quartier_m2'] = pd.to_numeric(quartiers['surface_quartier_m2'], errors='coerce')

# --- Achat: fusion de 75.csv (2021) + valeurs_foncieres.csv (2025)
DVF_FILES = [
    ROOT_DIR / 'data' / 'bronze' / 'immobilier' / '75.csv',
    ROOT_DIR / 'data' / 'bronze' / 'immobilier' / 'valeurs_foncieres.csv',
]


def load_dvf_source(csv_file: Path) -> pd.DataFrame:
    use_cols = ['date_mutation', 'valeur_fonciere', 'surface_reelle_bati', 'code_postal', 'type_local']
    df = pd.read_csv(csv_file, usecols=use_cols, parse_dates=['date_mutation'])
    df = df.dropna(subset=['date_mutation', 'valeur_fonciere', 'surface_reelle_bati', 'code_postal'])
    df = df[df['surface_reelle_bati'] > 9]
    df = df[df['valeur_fonciere'] > 10000]
    df = df[df['type_local'].isin(['Appartement', 'Maison'])]
    df['code_postal'] = df['code_postal'].astype(int).astype(str).str.zfill(5)
    df = df[df['code_postal'].str.startswith('75')]
    df['prix_m2'] = df['valeur_fonciere'] / df['surface_reelle_bati']
    q_low = df['prix_m2'].quantile(0.01)
    q_high = df['prix_m2'].quantile(0.99)
    df = df[(df['prix_m2'] >= q_low) & (df['prix_m2'] <= q_high)]
    df['annee'] = df['date_mutation'].dt.year
    df['arrondissement'] = df['code_postal'].str[-2:]
    return df[['annee', 'arrondissement', 'prix_m2']]


achat_all = pd.concat([load_dvf_source(p) for p in DVF_FILES], ignore_index=True)
achat_arr_annuel = (
    achat_all.groupby(['annee', 'arrondissement'], as_index=False)
    .agg(
        prix_m2_median=('prix_m2', 'median'),
        prix_m2_moyen=('prix_m2', 'mean'),
        nb_transactions=('prix_m2', 'size')
    )
)

# --- Location annuelle par arrondissement
loyers_path = ROOT_DIR / 'data' / 'bronze' / 'immobilier' / 'logement-encadrement-des-loyers.csv'
df_loyers = pd.read_csv(loyers_path, sep=';', encoding='utf-8-sig', low_memory=False)

col_annee = pick_column(df_loyers, ['annee', 'ann'])
col_arr = pick_column(df_loyers, ['secteurs geographiques', 'arrondissement'])
col_loyer_ref = pick_column(df_loyers, ['loyers de reference', 'loyer reference'])

if not all([col_annee, col_arr, col_loyer_ref]):
    raise ValueError('Colonnes loyers introuvables pour calcul annuel (annee/arrondissement/loyer_reference).')

loyers_work = df_loyers[[col_annee, col_arr, col_loyer_ref]].copy()
loyers_work[col_annee] = to_numeric(loyers_work[col_annee])
loyers_work[col_arr] = to_numeric(loyers_work[col_arr])
loyers_work[col_loyer_ref] = to_numeric(loyers_work[col_loyer_ref])
loyers_work = loyers_work.dropna(subset=[col_annee, col_arr, col_loyer_ref])

loyers_arr_annuel = (
    loyers_work.groupby([col_annee, col_arr], as_index=False)
    .agg(
        loyer_reference_median=(col_loyer_ref, 'median'),
        loyer_reference_moyen=(col_loyer_ref, 'mean'),
        nb_observations=(col_loyer_ref, 'size')
    )
    .rename(columns={col_annee: 'annee', col_arr: 'arrondissement'})
)
loyers_arr_annuel['annee'] = loyers_arr_annuel['annee'].astype(int)
loyers_arr_annuel['arrondissement'] = loyers_arr_annuel['arrondissement'].astype(int).astype(str).str.zfill(2)

# Base intermediaire achat/location sur annee + arrondissement (non exportee)
kpi_base_projection = achat_arr_annuel.merge(
    loyers_arr_annuel,
    on=['annee', 'arrondissement'],
    how='inner'
)

kpi_base_projection['kpi_comparaison_achat_location'] = (
    kpi_base_projection['prix_m2_median'] / (kpi_base_projection['loyer_reference_median'] * 12)
)

for c in ['prix_m2_median', 'prix_m2_moyen', 'loyer_reference_median', 'loyer_reference_moyen', 'kpi_comparaison_achat_location']:
    kpi_base_projection[c] = kpi_base_projection[c].round(4)

kpi_base_projection = kpi_base_projection.sort_values(['annee', 'arrondissement']).reset_index(drop=True)

print(f'Lignes base intermediaire: {len(kpi_base_projection):,}'.replace(',', ' '))
print('Annees disponibles:', sorted(kpi_base_projection['annee'].unique().tolist()))
display(kpi_base_projection.head(12))

Lignes KPI arrondissement annuel: 28
Annees disponibles: [2021, 2025]


,annee,arrondissement,prix_m2_median,prix_m2_moyen,nb_transactions,loyer_reference_median,loyer_reference_moyen,nb_observations,kpi_comparaison_achat_location
0,2021,01,14132.9231,43356.8352,370,30.40,30.7688,160,38.7416
1,2021,02,12352.9412,36549.1517,593,28.20,28.5562,448,36.5040
2,2021,03,13214.2857,36970.9064,781,27.80,27.9938,128,39.6112
3,2021,04,13534.4828,17053.5408,561,26.50,27.2531,352,42.5613
4,2021,05,13262.9310,27962.6983,1024,25.05,25.5156,256,44.1215
5,2021,06,15765.0502,22621.8596,928,26.80,26.7156,96,49.0207
6,2021,07,15277.7778,29788.9787,1118,25.75,25.7219,96,49.4426
7,2021,08,13529.4118,42608.0988,799,24.85,24.9094,32,45.3703
8,2021,09,12039.6389,25966.2826,1486,23.75,23.4500,96,42.2443
9,2021,10,10833.3333,19961.8006,2012,24.85,25.3375,192,36.3291


In [ ]:
# 2) Projection quartier + evolution
surface_arr = (
    quartiers.groupby('arrondissement', as_index=False)['surface_quartier_m2']
    .sum()
    .rename(columns={'surface_quartier_m2': 'surface_arrondissement_m2'})
)

quartiers = quartiers.merge(surface_arr, on='arrondissement', how='left')
quartiers['part_surface'] = quartiers['surface_quartier_m2'] / quartiers['surface_arrondissement_m2']

kpi_quartier_annuel = (
    kpi_base_projection.merge(
        quartiers[['code_insee_quartier', 'arrondissement', 'surface_quartier_m2', 'part_surface']],
        on='arrondissement',
        how='left'
    )
    .sort_values(['annee', 'arrondissement', 'code_insee_quartier'])
    .reset_index(drop=True)
)

# Volumes estimes par part de surface
kpi_quartier_annuel['nb_transactions_estime'] = (
    kpi_quartier_annuel['nb_transactions'] * kpi_quartier_annuel['part_surface']
).round().astype('Int64')
kpi_quartier_annuel['nb_observations_estime'] = (
    kpi_quartier_annuel['nb_observations'] * kpi_quartier_annuel['part_surface']
).round().astype('Int64')

# Evolution 2021 -> 2025 (si les deux annees existent)
annees = set(kpi_quartier_annuel['annee'].unique().tolist())
if {2021, 2025}.issubset(annees):
    base = kpi_quartier_annuel[kpi_quartier_annuel['annee'] == 2021].copy()
    target = kpi_quartier_annuel[kpi_quartier_annuel['annee'] == 2025].copy()

    cols_keep = [
        'arrondissement', 'code_insee_quartier',
        'prix_m2_median', 'loyer_reference_median', 'kpi_comparaison_achat_location'
    ]
    base = base[cols_keep].rename(columns={
        'prix_m2_median': 'prix_m2_median_2021',
        'loyer_reference_median': 'loyer_reference_median_2021',
        'kpi_comparaison_achat_location': 'kpi_comparaison_achat_location_2021'
    })
    target = target[cols_keep].rename(columns={
        'prix_m2_median': 'prix_m2_median_2025',
        'loyer_reference_median': 'loyer_reference_median_2025',
        'kpi_comparaison_achat_location': 'kpi_comparaison_achat_location_2025'
    })

    kpi_quartier_evolution = base.merge(
        target,
        on=['arrondissement', 'code_insee_quartier'],
        how='inner'
    )

    kpi_quartier_evolution['evolution_achat_pct'] = (
        (kpi_quartier_evolution['prix_m2_median_2025'] - kpi_quartier_evolution['prix_m2_median_2021'])
        / kpi_quartier_evolution['prix_m2_median_2021'] * 100
    ).round(4)

    kpi_quartier_evolution['evolution_loyer_pct'] = (
        (kpi_quartier_evolution['loyer_reference_median_2025'] - kpi_quartier_evolution['loyer_reference_median_2021'])
        / kpi_quartier_evolution['loyer_reference_median_2021'] * 100
    ).round(4)

    kpi_quartier_evolution['evolution_kpi_comparaison_pct'] = (
        (kpi_quartier_evolution['kpi_comparaison_achat_location_2025'] - kpi_quartier_evolution['kpi_comparaison_achat_location_2021'])
        / kpi_quartier_evolution['kpi_comparaison_achat_location_2021'] * 100
    ).round(4)
else:
    kpi_quartier_evolution = pd.DataFrame()

kpi_quartier_annuel = kpi_quartier_annuel[
    [
        'annee', 'arrondissement', 'code_insee_quartier',
        'prix_m2_median', 'prix_m2_moyen', 'nb_transactions',
        'loyer_reference_median', 'loyer_reference_moyen', 'nb_observations',
        'kpi_comparaison_achat_location',
        'surface_quartier_m2', 'part_surface',
        'nb_transactions_estime', 'nb_observations_estime'
    ]
]

print(f'Lignes KPI quartier annuel: {len(kpi_quartier_annuel):,}'.replace(',', ' '))
if not kpi_quartier_evolution.empty:
    print(f'Lignes KPI quartier evolution 2021-2025: {len(kpi_quartier_evolution):,}'.replace(',', ' '))
else:
    print('Evolution 2021-2025 non calculee (annees indisponibles dans la table annuelle).')

display(kpi_quartier_annuel.head(12))

Lignes KPI quartier annuel: 112
Lignes KPI quartier evolution 2021-2025: 56


,annee,arrondissement,prix_m2_median,prix_m2_moyen,nb_transactions,loyer_reference_median,loyer_reference_moyen,nb_observations,kpi_comparaison_achat_location,code_insee_quartier,nom_quartier,surface_quartier_m2,part_surface,geometry_xy,nb_transactions_estime,nb_observations_estime
0,2021,01,14132.9231,43356.8352,370,30.4,30.7688,160,38.7416,7510101,Saint-Germain-l'Auxerrois,869000.664564,0.476266,"48.86065013520993, 2.3349103292801994",176,76
1,2021,01,14132.9231,43356.8352,370,30.4,30.7688,160,38.7416,7510102,Halles,412458.496330,0.226053,"48.86228910809422, 2.3448988583110166",84,36
2,2021,01,14132.9231,43356.8352,370,30.4,30.7688,160,38.7416,7510103,Palais-Royal,273696.793301,0.150003,"48.86465997810256, 2.3363089189653063",56,24
3,2021,01,14132.9231,43356.8352,370,30.4,30.7688,160,38.7416,7510104,Place-Vendôme,269456.780599,0.147679,"48.867018590627694, 2.328581664931248",55,24
4,2021,02,12352.9412,36549.1517,593,28.2,28.5562,448,36.5040,7510201,Gaillon,188012.203850,0.189690,"48.869306638088744, 2.3334318076581146",112,85
5,2021,02,12352.9412,36549.1517,593,28.2,28.5562,448,36.5040,7510202,Vivienne,243550.770623,0.245725,"48.86910019977125, 2.3394607437483312",146,110
6,2021,02,12352.9412,36549.1517,593,28.2,28.5562,448,36.5040,7510203,Mail,278142.585084,0.280625,"48.868008337402706, 2.3446991274274604",166,126
7,2021,02,12352.9412,36549.1517,593,28.2,28.5562,448,36.5040,7510204,Bonne-Nouvelle,281448.206577,0.283960,"48.8671501183107, 2.3500801904088187",168,127
8,2021,03,13214.2857,36970.9064,781,27.8,27.9938,128,39.6112,7510301,Arts-et-Métiers,318087.740454,0.271665,"48.86647028948414, 2.3570831310567835",212,35
9,2021,03,13214.2857,36970.9064,781,27.8,27.9938,128,39.6112,7510302,Enfants-Rouges,271750.323937,0.232090,"48.86388739200144, 2.363123300986952",181,30


In [ ]:
# 3) Export (sortie quartier uniquement)
output_dir = Path('../../data/outputs')
output_dir.mkdir(parents=True, exist_ok=True)

output_quartier = output_dir / 'kpi_comparaison_achat_location_quartier_estime.csv'
kpi_quartier_annuel.to_csv(output_quartier, index=False)

print(f'Fichier cree: {output_quartier}')
print(f'Nombre de lignes quartier estime: {len(kpi_quartier_annuel):,}'.replace(',', ' '))

Fichier cree: ..\..\data\outputs\kpi_comparaison_achat_location_arrondissement.csv
Fichier cree: ..\..\data\outputs\kpi_comparaison_achat_location_quartier_estime.csv
Nombre de lignes arrondissement: 28
Nombre de lignes quartier estime: 112
